#  Heavy ECG Classification (Full Model)
Deep ResNet + Multi-Scale + SE Attention


##  Install Dependencies

In [1]:
!pip install imbalanced-learn

##  Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

2026-04-08 12:59:04.356532: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775653144.586130      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775653144.652811      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775653145.270042      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775653145.270083      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775653145.270086      24 computation_placer.cc:177] computation placer alr

TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


##  Load Dataset

In [3]:
DATA_DIR = "/kaggle/input/datasets/shayanfazeli/heartbeat"

train_df = pd.read_csv(os.path.join(DATA_DIR, "mitbih_train.csv"), header=None)
test_df  = pd.read_csv(os.path.join(DATA_DIR, "mitbih_test.csv"), header=None)

X_train = train_df.iloc[:, :-1].values
y_train = train_df.iloc[:, -1].values
X_test  = test_df.iloc[:, :-1].values
y_test  = test_df.iloc[:, -1].values

y_train = np.where(y_train == 0, 0, 1)
y_test  = np.where(y_test == 0, 0, 1)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (87554, 187) Test: (21892, 187)


##  SMOTE

In [4]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

##  Preprocessing

In [5]:
X_train_sm = X_train_sm.reshape(-1,187,1).astype(np.float32)
X_test = X_test.reshape(-1,187,1).astype(np.float32)

X_train_sm = (X_train_sm - X_train_sm.min())/(X_train_sm.max()-X_train_sm.min()+1e-8)
X_test = (X_test - X_test.min())/(X_test.max()-X_test.min()+1e-8)

##  Model Architecture (Full Heavy Model)

In [6]:
def squeeze_excite_block(inputs, ratio=16):
    channels = inputs.shape[-1]
    se = layers.GlobalAveragePooling1D()(inputs)
    se = layers.Dense(channels // ratio, activation='relu')(se)
    se = layers.Dense(channels, activation='sigmoid')(se)
    se = layers.Reshape((1, channels))(se)
    return layers.Multiply()([inputs, se])

def residual_block(x, filters, stride=1):
    shortcut = x
    x = layers.Conv1D(filters, 3, strides=stride, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv1D(filters, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = squeeze_excite_block(x)
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    x = layers.Add()([x, shortcut])
    return layers.Activation('relu')(x)

def multi_scale_block(x, filters):
    b1 = layers.Conv1D(filters//4,3,padding='same')(x)
    b2 = layers.Conv1D(filters//4,5,padding='same')(x)
    b3 = layers.Conv1D(filters//4,7,padding='same')(x)
    b4 = layers.MaxPooling1D(3,strides=1,padding='same')(x)
    b4 = layers.Conv1D(filters//4,1,padding='same')(b4)
    return layers.Concatenate()([b1,b2,b3,b4])

def build_model():
    inputs = layers.Input(shape=(187,1))
    x = multi_scale_block(inputs,64)
    x = layers.MaxPooling1D(2)(x)

    x = residual_block(x,64)
    x = residual_block(x,64)
    x = layers.MaxPooling1D(2)(x)

    x = residual_block(x,128)
    x = residual_block(x,128)
    x = residual_block(x,128)
    x = layers.MaxPooling1D(2)(x)

    x = residual_block(x,256)
    x = residual_block(x,256)
    x = residual_block(x,256)
    x = residual_block(x,256)
    x = layers.MaxPooling1D(2)(x)

    x = residual_block(x,512)
    x = residual_block(x,512)
    x = residual_block(x,512)

    avg = layers.GlobalAveragePooling1D()(x)
    mx = layers.GlobalMaxPooling1D()(x)
    x = layers.Concatenate()([avg,mx])

    x = layers.Dense(512,activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256,activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128,activation='relu')(x)

    outputs = layers.Dense(1,activation='sigmoid')(x)
    return Model(inputs,outputs)

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(0.001),
              loss='binary_crossentropy',
              metrics=['accuracy'])
model.summary()

I0000 00:00:1775653180.492433      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775653180.498442      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 187, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 187, 1)    │          0 │ input_layer[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 187, 16)   │         64 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 187, 16)   │         96 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 187, 16)   │        128 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 187, 16)   │         32 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 187, 64)   │          0 │ conv1d[0][0],     │
│ (Concatenate)       │                   │            │ conv1d_1[0][0],   │
│                     │                   │            │ conv1d_2[0][0],   │
│                     │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 93, 64)    │          0 │ concatenate[0][0] │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 93, 64)    │     12,352 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 93, 64)    │        256 │ conv1d_4[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 93, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 93, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 93, 64)    │        256 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4)         │        260 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │        320 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1, 64)     │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply (Multiply) │ (None, 93, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 93, 64)    │          0 │ multiply[0][0],   │
│                     │                   │            │ max_pooling1d_1[

 Total params: 7,157,633 (27.30 MB)

 Trainable params: 7,143,553 (27.25 MB)

 Non-trainable params: 14,080 (55.00 KB)

##  Training 

In [7]:
callbacks = [
    EarlyStopping(patience=10, restore_best_weights=True),
    ReduceLROnPlateau(patience=5)
]

history = model.fit(
    X_train_sm, y_train_sm,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=128,
    callbacks=callbacks
)

Epoch 1/100


I0000 00:00:1775653205.456663      83 service.cc:152] XLA service 0x7ec6a0084360 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775653205.456698      83 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775653205.456702      83 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775653210.002683      83 cuda_dnn.cc:529] Loaded cuDNN version 91002


   2/1133 ━━━━━━━━━━━━━━━━━━━━ 1:25 75ms/step - accuracy: 0.5742 - loss: 1.4296   

I0000 00:00:1775653225.175596      83 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1133/1133 ━━━━━━━━━━━━━━━━━━━━ 112s 62ms/step - accuracy: 0.9244 - loss: 0.2109 - val_accuracy: 0.9791 - val_loss: 0.0749 - learning_rate: 0.0010
Epoch 2/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 50s 44ms/step - accuracy: 0.9781 - loss: 0.0646 - val_accuracy: 0.9818 - val_loss: 0.0654 - learning_rate: 0.0010
Epoch 3/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 50s 44ms/step - accuracy: 0.9847 - loss: 0.0452 - val_accuracy: 0.9745 - val_loss: 0.0763 - learning_rate: 0.0010
Epoch 4/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 50s 44ms/step - accuracy: 0.9872 - loss: 0.0383 - val_accuracy: 0.9850 - val_loss: 0.0493 - learning_rate: 0.0010
Epoch 5/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 49s 43ms/step - accuracy: 0.9894 - loss: 0.0313 - val_accuracy: 0.9784 - val_loss: 0.0623 - learning_rate: 0.0010
Epoch 6/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 49s 43ms/step - accuracy: 0.9913 - loss: 0.0260 - val_accuracy: 0.9854 - val_loss: 0.0496 - learning_rate: 0.0010
Epoch 7/100
1133/1133 ━━━━━━━━━━━━━━━━━━━━ 49s 43ms/step - accuracy: 

##  Evaluation

In [8]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print(classification_report(y_test, y_pred))

685/685 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     18118
           1       0.96      0.95      0.96      3774

    accuracy                           0.99     21892
   macro avg       0.98      0.97      0.97     21892
weighted avg       0.98      0.99      0.98     21892



In [9]:
# Export Model to TFLite (float32) and H5 formats
import tensorflow as tf

# Save as H5 format
h5_path = "ecg_model_7M_params.h5"
model.save(h5_path)
print(f"Model saved as H5: {h5_path}")

# Convert to TFLite (float32)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = []  # Keep float32 precision
tflite_model = converter.convert()

# Save TFLite model
tflite_path = "ecg_model_7M_params_float32.tflite"
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"Model saved as TFLite (float32): {tflite_path}")

# Check file sizes
import os
h5_size = os.path.getsize(h5_path) / (1024 * 1024)  # MB
tflite_size = os.path.getsize(tflite_path) / (1024 * 1024)  # MB

print(f"\nFile Sizes:")
print(f"H5 Model: {h5_size:.2f} MB")
print(f"TFLite Model: {tflite_size:.2f} MB")
print(f"Compression ratio: {h5_size/tflite_size:.2f}x")

Model saved as H5: ecg_model_7M_params.h5
INFO:tensorflow:Assets written to: /tmp/tmpxxaaejy6/assets


INFO:tensorflow:Assets written to: /tmp/tmpxxaaejy6/assets


Saved artifact at '/tmp/tmpxxaaejy6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 187, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139396815010320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396815011472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136453520: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136452752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396815010896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136453136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136452944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136454480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136454288: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136453328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139396136457360: Te

W0000 00:00:1775653962.497644      24 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1775653962.497679      24 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1775653962.627646      24 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled


Model saved as TFLite (float32): ecg_model_7M_params_float32.tflite

File Sizes:
H5 Model: 82.44 MB
TFLite Model: 27.26 MB
Compression ratio: 3.02x


In [10]:
# MODEL PERFORMANCE SUMMARY
print("="*60)
print("ECG 7M PARAMS MODEL - FINAL RESULTS")
print("="*60)
print(f"Total Parameters:     {model.count_params():,}")
print(f"Model Size:            7,157,633 parameters (27.30 MB)")
print(f"Test Accuracy:        0.9885 (98.85%)")
print(f"Test Loss:            0.1131")
print()
print("Key Features:")
print("- Deep ResNet architecture with residual blocks")
print("- Multi-scale feature extraction")
print("- Squeeze-and-Excitation attention")
print("- SMOTE balanced dataset")
print("- TFLite export ready")
print("="*60)

ECG 7M PARAMS MODEL - FINAL RESULTS
Total Parameters:     7,157,633
Model Size:            7,157,633 parameters (27.30 MB)
Test Accuracy:        0.9885 (98.85%)
Test Loss:            0.1131

Key Features:
- Deep ResNet architecture with residual blocks
- Multi-scale feature extraction
- Squeeze-and-Excitation attention
- SMOTE balanced dataset
- TFLite export ready
